In [1]:
### TUGAS MANDIRI SOAL 1

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum, count, window
import os
import time
import shutil


spark = SparkSession.builder \
    .appName("Tugas10_Kopi_Nusantara") \
    .getOrCreate() # sparksession

# Mengatur jumlah shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")

26/11/11 21:36:16 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/11 21:36:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 21:36:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 21:36:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat.


In [13]:
# Bikin skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True), 
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membuat folder simulasi streaming
folder_stream = "tugas10_stream"
os.makedirs(folder_stream, exist_ok=True)

# Membaca data streaming
df_stream = spark.readStream \
    .option("header", True) \
    .schema(skema_pesanan) \
    .csv(folder_stream)

print("Skema streaming:")
df_stream.printSchema()

Skema streaming:
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



In [15]:
### TUGAS MANDIRI SOAL 2

# menghitung total
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan"))
    
# data dai cabang yang namanya sama dikumpulin baru dihitung toal
agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# untuk menjalankan streaming
query1 = agregasi_cabang.writeStream \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .outputMode("complete") \
    .start()

print("Streaming agregasi cabang berhasil dijalankan.")

Streaming agregasi cabang berhasil dijalankan.


26/11/11 21:59:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-a739f4a5-f929-4d05-8703-ba7287428898. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/11 21:59:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [16]:
### TUGAS MANDIRI SOAL 3

# Membuat agregasi window 2 menit.
df_stream_waktu = spark.readStream \
    .option("header", True) \
    .schema(skema_pesanan) \
    .csv(folder_stream)

df_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

# window 2 menit
agregasi_waktu = df_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    sum("total_harga").alias("total_pendapatan")
)

query2 = agregasi_waktu.writeStream \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .outputMode("complete") \
    .start()

print("Streaming Agregasi waktu berhasil dijalankan.")

Streaming Agregasi waktu berhasil dijalankan.


26/11/11 21:59:27 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e0e48853-8428-476c-ba93-5262ea341bf3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/11 21:59:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [17]:
### TUGAS MANDIRI SOAL 4

# MENGIRIM BATCH 1

data_batch1 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P001,Kopi Nusantara Temanggung,Menu-1,2,50000,2026-11-11 10:00:10
P002,Kopi Nusantara Bali,Menu-2,1,80000,2026-11-11 10:00:40
"""

with open(os.path.join(folder_stream, "batch1.csv"), "w") as file:
    file.write(data_batch1)

time.sleep(3)
query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
|Kopi Nusantara Te...|          100000|             1|
| Kopi Nusantara Bali|           80000|             1|
+--------------------+----------------+--------------+



In [18]:
# MENGIRIM BATCH 2

data_batch2 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P003,Kopi Nusantara Semarang,Menu-3,3,25000,2026-11-11 10:01:10
P004,Kopi Nusantara Magelang,Menu-4,2,30000,2026-11-11 10:02:10
"""

with open(os.path.join(folder_stream, "batch2.csv"), "w") as file:
    file.write(data_batch2)

time.sleep(3)
query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
|Kopi Nusantara Te...|          100000|             1|
| Kopi Nusantara Bali|           80000|             1|
|Kopi Nusantara Se...|           75000|             1|
|Kopi Nusantara Ma...|           60000|             1|
+--------------------+----------------+--------------+



In [19]:
# MENGIRIM BATCH 3

data_batch3 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P005,Kopi Nusantara Temanggung,Menu-1,2,18000,2026-11-11 10:02:30
P006,Kopi Nusantara Semarang,Menu-2,1,22000,2026-11-11 10:03:00
"""

with open(os.path.join(folder_stream, "batch3.csv"), "w") as file:
    file.write(data_batch3)

time.sleep(3)
query1.processAllAvailable()
query2.processAllAvailable()

spark.sql("SELECT * FROM agregasi_cabang").show()

+--------------------+----------------+--------------+
|         nama_cabang|total_pendapatan|jumlah_pesanan|
+--------------------+----------------+--------------+
|Kopi Nusantara Te...|          136000|             2|
|Kopi Nusantara Se...|           97000|             2|
| Kopi Nusantara Bali|           80000|             1|
|Kopi Nusantara Ma...|           60000|             1|
+--------------------+----------------+--------------+



In [20]:
## AGREGASI WINDOW

spark.sql("SELECT * FROM agregasi_waktu").show(truncate=False)

+------------------------------------------+----------------+
|window                                    |total_pendapatan|
+------------------------------------------+----------------+
|{2026-11-11 10:02:00, 2026-11-11 10:04:00}|118000          |
|{2026-11-11 10:00:00, 2026-11-11 10:02:00}|255000          |
+------------------------------------------+----------------+



In [21]:
### SINK

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming file sink berhasil dijalankan.")

Streaming file sink berhasil dijalankan.


26/11/11 22:05:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

In [22]:
data_batch4 = """pesanan_id,nama_cabang,nama_menu,qty,harga_satuan,waktu_pesan
P007,Kopi Nusantara Bali,Menu-5,1,35000,2026-11-11 10:03:20
P008,Kopi Nusantara Magelang,Menu-3,2,25000,2026-11-11 10:03:40
"""

with open(os.path.join(folder_stream, "batch4.csv"), "w") as file:
    file.write(data_batch4)

time.sleep(3)
query3.processAllAvailable()

print("Batch baru berhasil dikirim.")

Batch baru berhasil dikirim.


In [23]:
!ls tugas10_output

part-00000-88901455-10e3-4f25-a018-33993f39f818-c000.snappy.parquet
part-00000-a777849f-11ae-4a2f-951a-7f419122cc24-c000.snappy.parquet
part-00001-ec4e326a-055b-40cd-981b-da6b7bb33428-c000.snappy.parquet
part-00002-f666bad1-3a32-449d-ab73-138536bf2c2f-c000.snappy.parquet
_spark_metadata


In [24]:
df_parquet = spark.read.parquet("tugas10_output")

df_parquet.show(truncate=False)

+----------+-------------------------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang              |nama_menu|qty|harga_satuan|waktu_pesan        |
+----------+-------------------------+---------+---+------------+-------------------+
|P005      |Kopi Nusantara Temanggung|Menu-1   |2  |18000       |2026-11-11 10:02:30|
|P006      |Kopi Nusantara Semarang  |Menu-2   |1  |22000       |2026-11-11 10:03:00|
|P003      |Kopi Nusantara Semarang  |Menu-3   |3  |25000       |2026-11-11 10:01:10|
|P004      |Kopi Nusantara Magelang  |Menu-4   |2  |30000       |2026-11-11 10:02:10|
|P001      |Kopi Nusantara Temanggung|Menu-1   |2  |50000       |2026-11-11 10:00:10|
|P002      |Kopi Nusantara Bali      |Menu-2   |1  |80000       |2026-11-11 10:00:40|
|P007      |Kopi Nusantara Bali      |Menu-5   |1  |35000       |2026-11-11 10:03:20|
|P008      |Kopi Nusantara Magelang  |Menu-3   |2  |25000       |2026-11-11 10:03:40|
+----------+-------------------------+---------+---+--